In [13]:
# Housekeeping
import sqlite3
import csv
import urllib.request
import os

In [4]:
# GitHub raw URLs for the three CSV files
BASE_URL = "https://raw.githubusercontent.com/nawanglhantso/CIS-3120/main/"
BOOK_URL   = BASE_URL + "Book.csv"
MEMBER_URL = BASE_URL + "Member.csv"
LOAN_URL   = BASE_URL + "Loan.csv"

In [14]:
# Local paths in the Colab /content directory
BOOK_PATH = "/Users/nawanglhantso/Downloads/Book.csv"
MEMBER_PATH = "/Users/nawanglhantso/Downloads/Member.csv"
LOAN_PATH = "/Users/nawanglhantso/Downloads/Loan.csv"
DB_PATH = "library.db"

In [15]:
import requests

In [16]:
for url, path in [(BOOK_URL, BOOK_PATH), (MEMBER_URL, MEMBER_PATH), (LOAN_URL, LOAN_PATH)]:
    r = requests.get(url)
    with open(path, 'wb') as f:
        f.write(r.content)
    print(f"Downloaded: {path}")

Downloaded: /Users/nawanglhantso/Downloads/Book.csv
Downloaded: /Users/nawanglhantso/Downloads/Member.csv
Downloaded: /Users/nawanglhantso/Downloads/Loan.csv


In [20]:
import sqlite3
import os
import csv

In [18]:
# connect to database
conn = sqlite3.connect(DB_PATH)

# enable foreign keys
conn.execute("PRAGMA foreign_keys = ON;")

# create tables
conn.execute("""
CREATE TABLE IF NOT EXISTS Book (
    callNo  TEXT    NOT NULL,
    title   TEXT    NOT NULL,
    author  TEXT    NOT NULL,
    PRIMARY KEY (callNo)
);
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS Member (
    id          INTEGER NOT NULL,
    firstname   TEXT    NOT NULL,
    lastName    TEXT    NOT NULL,
    PRIMARY KEY (id)
);
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS Loan (
    callNo        TEXT    NOT NULL,
    id            INTEGER NOT NULL,
    dateBorrowed  TEXT    NOT NULL,
    dateReturned  TEXT,
    dateDue       TEXT    NOT NULL,
    PRIMARY KEY (callNo, id, dateBorrowed),
    FOREIGN KEY (callNo) REFERENCES Book(callNo),
    FOREIGN KEY (id) REFERENCES Member(id)
);
""")

conn.commit()

print("Tables created:")
print(conn.execute("SELECT name FROM sqlite_master WHERE type='table';").fetchall())

Tables created:
[('Book',), ('Member',), ('Loan',)]


In [21]:
with open(BOOK_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        conn.execute(
            "INSERT INTO Book (callNo, title, author) VALUES (?, ?, ?);",
            (row["callNo"], row["title"], row["author"])
        )

conn.commit()
print("Book rows loaded:", conn.execute("SELECT COUNT(*) FROM Book;").fetchone()[0])

Book rows loaded: 11


In [22]:
with open(MEMBER_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        conn.execute(
            "INSERT INTO Member (id, firstname, lastName) VALUES (?, ?, ?);",
            (int(row["id"]), row["firstname"], row["lastName"])
        )

conn.commit()
print("Member rows loaded:", conn.execute("SELECT COUNT(*) FROM Member;").fetchone()[0])

Member rows loaded: 4


In [23]:
with open(LOAN_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        date_returned = row["dateReturned"] if row["dateReturned"].strip() else None

        conn.execute("""
            INSERT INTO Loan (callNo, id, dateBorrowed, dateReturned, dateDue)
            VALUES (?, ?, ?, ?, ?);
        """, (
            row["callNo"],
            int(row["id"]),
            row["dateBorrowed"],
            date_returned,
            row["dateDue"]
        ))

conn.commit()
print("Loan rows loaded:", conn.execute("SELECT COUNT(*) FROM Loan;").fetchone()[0])

Loan rows loaded: 4


In [24]:
query1 = """
SELECT *
FROM Book
ORDER BY author;
"""

rows = conn.execute(query1).fetchall()
for row in rows:
    print(row)

('R 487 T35 1967', 'Medicine in medieval England.', 'Charles H Talbot')
('QA 76.9 D26H39 1996', 'Data model patterns : conventions of thought', 'David Hay')
('CB 351 M293 1983', 'Atlas of medieval Europe', 'Donald Matthew')
('HQ 1143 P68 1975', 'Medieval women', 'Eileen Power')
('PC 14 V48 1965', 'Medieval miscellany', 'Frederick Whitehead')
('QA 76.73 S67C435 2004', "Joe Celko's Trees and hierarchies in SQL for smarties", 'Joe Celko')
('QA 76.73 S67C46 1997', "Joe Celko's SQL puzzles & answers", 'Joe Celko')
('QA 76.9 D35C45 1999', "Joe Celko's data & databases : concepts in practice", 'Joe Celko')
('R 141 E45 2006', 'Medieval medicine and the plague', 'Lynne Elliott')
('QA 76.9 D26H355 2008', 'Information modeling and relational databases', 'T A Halpin')
('QA 76.76 A65P76 2011', 'Programming Android', 'Zigurd R Mednieks')


In [25]:
query2 = """
SELECT b.title, m.firstname, m.lastName
FROM Loan l
JOIN Book b ON l.callNo = b.callNo
JOIN Member m ON l.id = m.id
WHERE l.dateReturned IS NULL;
"""

rows = conn.execute(query2).fetchall()
for row in rows:
    print(row)

("Joe Celko's SQL puzzles & answers", 'David', 'Martin')
('Medieval medicine and the plague', 'David', 'Martin')


In [26]:
query3 = """
SELECT m.firstname, m.lastName, l.dateBorrowed, l.dateDue, l.dateReturned
FROM Loan l
JOIN Member m ON l.id = m.id
WHERE l.callNo = 'R 141 E45 2006'
ORDER BY l.dateBorrowed ASC;
"""

rows = conn.execute(query3).fetchall()
for row in rows:
    print(row)

('Betty', 'Freeman', '4/1/2014 0:00', '4/15/2014 0:00', '4/15/2014 0:00')
('David', 'Martin', '4/30/2014 0:00', '5/14/2014 0:00', None)


In [27]:
query4 = """
SELECT m.id, m.firstname, m.lastName
FROM Member m
LEFT JOIN Loan l ON m.id = l.id
WHERE l.id IS NULL;
"""

rows = conn.execute(query4).fetchall()
for row in rows:
    print(row)

(4, 'John', 'Martin')


In [28]:
query5 = """
SELECT m.firstname, m.lastName, COUNT(l.callNo) AS total_loans
FROM Member m
LEFT JOIN Loan l ON m.id = l.id
GROUP BY m.id, m.firstname, m.lastName
ORDER BY total_loans DESC, m.lastName, m.firstname;
"""

rows = conn.execute(query5).fetchall()
for row in rows:
    print(row)

('David', 'Martin', 2)
('Betty', 'Freeman', 1)
('John', 'Smith', 1)
('John', 'Martin', 0)


In [29]:
query6 = """
SELECT m.firstname, m.lastName, COUNT(*) AS books_outstanding
FROM Loan l
JOIN Member m ON l.id = m.id
WHERE l.dateReturned IS NULL
GROUP BY m.id, m.firstname, m.lastName
ORDER BY books_outstanding DESC, m.lastName, m.firstname;
"""

rows = conn.execute(query6).fetchall()
for row in rows:
    print(row)

('David', 'Martin', 2)


 Summary

One data quality issue in this dataset is that some values in the `dateReturned` column are blank, which required converting them to NULL when loading the data into SQLite. Another observation is that all date fields are stored as text rather than in a standardized date format, which could make date comparisons less reliable in larger systems. A limitation of this dataset is its small size, since it includes only a few books, members, and loan records. In a real library system, additional features such as tracking multiple copies of books, overdue fines, reservations, and book categories would be necessary.

In [30]:
conn.close()
print("Database connection closed.")

Database connection closed.


In [31]:
git add .
git commit -m "Complete Library DB assignment"
git push

SyntaxError: invalid syntax (4016445626.py, line 1)